# APOC - Awesome Procedures on Cypher

## Taller Educativo sobre Procedimientos Avanzados en Neo4j

Autor: mCárdenas 2025

### Objetivos de Aprendizaje

En este cuaderno aprenderás:

1. **APOC Basics**: Qué es APOC y cómo utilizarlo
2. **Utilidades**: Metadatos, fechas, texto, colecciones
3. **Path Expansion**: Expandir caminos con criterios complejos
4. **Batch Processing**: Operaciones en lotes eficientes
5. **Cypher Dinámico**: Construir y ejecutar consultas dinámicamente
6. **Import/Export**: Cargar y exportar datos
7. **Aplicación a Fraude**: Usar APOC para análisis de fraude de IVA



## 1. Introducción a APOC

### 1.1 ¿Qué es APOC?

**APOC (Awesome Procedures on Cypher)** es la biblioteca estándar de procedimientos y funciones para Neo4j.

**Características**:
- 450+ procedimientos y funciones
- Operaciones que Cypher no puede hacer nativamente
- Utilidades, algoritmos, I/O, refactoring
- Mantenido por Neo4j Labs

### 1.2 Categorías Principales

- **apoc.meta**: Metadatos del grafo
- **apoc.date**: Manipulación de fechas
- **apoc.text**: Operaciones con strings
- **apoc.coll**: Colecciones y listas
- **apoc.path**: Expansión de caminos
- **apoc.periodic**: Operaciones en lotes
- **apoc.cypher**: Cypher dinámico
- **apoc.load/export**: I/O de datos
- **apoc.refactor**: Reestructuración

### 1.3 APOC vs GDS

| Característica | APOC | GDS |
|----------------|------|-----|
| Algoritmos de grafos | Deprecados (v5+) | Optimizados |
| Utilidades | ✅ Extenso | Limitado |
| I/O Datos | ✅ Sí | No |
| Machine Learning | No | ✅ Sí |
| Cuando usar | Utilidades, I/O | Algoritmos |



## 2. Configuración y Conexión

In [ ]:
# Importar bibliotecas
from neo4j import GraphDatabase
import pandas as pd
import json

# Importar módulos de consultas
import sys
sys.path.append('.')
from consultas_apoc import *

print("✓ Bibliotecas importadas")

In [ ]:
# Configuración de conexión
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "abc123456"  # CAMBIAR
NEO4J_DATABASE = "fraudedb"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

def ejecutar_consulta(query, parametros=None):
    with driver.session(database=NEO4J_DATABASE) as session:
        resultado = session.run(query, parametros or {})
        return [record.data() for record in resultado]

# Verificar conexión
try:
    ejecutar_consulta("RETURN 'Conectado' AS status")
    print("✓ Conectado a Neo4j")
except Exception as e:
    print(f"✗ Error: {e}")

### 2.1 Verificar APOC

In [ ]:
# Verificar versión de APOC
resultado = ejecutar_consulta("RETURN apoc.version() AS version")
print(f"APOC Version: {resultado[0]['version']}")



## 3. Utilidades y Funciones Básicas

### 3.1 Metadatos del Grafo

In [ ]:
# Ver esquema del grafo
query = APOC_METADATOS["queries"]["schema"]
schema = ejecutar_consulta(query)

print("Esquema del grafo:")
for item in schema[:3]:
    print(json.dumps(item, indent=2, ensure_ascii=False))

In [ ]:
# Estadísticas del grafo
query = APOC_METADATOS["queries"]["stats"]
stats = ejecutar_consulta(query)
print("Estadísticas:")
print(json.dumps(stats[0], indent=2, ensure_ascii=False))

### 3.2 Manipulación de Texto

In [ ]:
# Ejemplos de funciones de texto
for ejemplo in APOC_TEXT["ejemplos"]:
    resultado = ejecutar_consulta(ejemplo)
    print(f"Query: {ejemplo}")
    print(f"Resultado: {resultado[0]}")
    print()

### 3.3 Operaciones con Colecciones

In [ ]:
# Ejemplos de colecciones
for ejemplo in APOC_COLLECTIONS["ejemplos"]:
    resultado = ejecutar_consulta(ejemplo)
    print(f"Query: {ejemplo}")
    print(f"Resultado: {resultado[0]}")
    print()

## 4. Path Expansion - Expansión de Caminos

**apoc.path.expandConfig** permite expandir caminos con criterios muy específicos.

### 4.1 Encontrar Caminos Entre Empresas

In [ ]:
# Primero, obtener nombres de 2 empresas para el ejemplo
empresas = ejecutar_consulta("""
MATCH (e:Empresa)
RETURN e.nombre AS nombre
ORDER BY rand()
LIMIT 2
""")

if len(empresas) >= 2:
    empresa_inicio = empresas[0]['nombre']
    empresa_fin = empresas[1]['nombre']
    
    print(f"Buscando caminos de '{empresa_inicio}' a '{empresa_fin}'...")
    
    query = f"""
    MATCH (inicio:Empresa {{nombre: '{empresa_inicio}'}}),
          (fin:Empresa {{nombre: '{empresa_fin}'}})
    CALL apoc.path.expandConfig(inicio, {{
        relationshipFilter: 'EMITE_FACTURA>',
        labelFilter: '+Empresa',
        minLevel: 1,
        maxLevel: 3,
        endNodes: [fin],
        uniqueness: 'NODE_PATH'
    }})
    YIELD path
    RETURN [n IN nodes(path) | n.nombre] AS camino, length(path) AS longitud
    ORDER BY longitud
    LIMIT 5
    """
    
    caminos = ejecutar_consulta(query)
    for c in caminos:
        print(f"  Camino ({c['longitud']} saltos): {' → '.join(c['camino'])}")
else:
    print("No hay suficientes empresas en la BD")

### 4.2 Expandir Subgrafo desde Empresa Fantasma

In [ ]:
# Expandir desde una empresa fantasma
query = """
MATCH (e:Empresa) WHERE e.es_fantasma = true
WITH e LIMIT 1
CALL apoc.path.subgraphAll(e, {
    relationshipFilter: 'EMITE_FACTURA',
    maxLevel: 2
})
YIELD nodes, relationships
RETURN 
    e.nombre AS empresa_origen,
    size(nodes) AS num_nodos,
    size(relationships) AS num_relaciones
"""

resultado = ejecutar_consulta(query)
if resultado:
    r = resultado[0]
    print(f"Subgrafo desde '{r['empresa_origen']}':")
    print(f"  Nodos alcanzados: {r['num_nodos']}")
    print(f"  Relaciones: {r['num_relaciones']}")

## 5. Periodic Iterate - Operaciones en Lotes

**apoc.periodic.iterate** permite procesar grandes volúmenes de datos eficientemente.

In [ ]:
# Ejemplo: Marcar empresas procesadas en lotes
query = APOC_PERIODIC["example_update_batch"]

print("Ejecutando actualización en lotes...")
resultado = ejecutar_consulta(query)
print(f"Batches procesados: {resultado[0]['batches']}")
print(f"Total actualizado: {resultado[0]['total']}")

## 6. Cypher Dinámico

Ejecutar consultas construidas dinámicamente.

In [ ]:
# Ejecutar query dinámica
query = APOC_CYPHER_DINAMICO["example_run"]
resultado = ejecutar_consulta(query)
print(f"Empresas en España: {resultado[0]['empresas_espanolas']}")

In [ ]:
# Análisis por país en paralelo
query = APOC_CYPHER_DINAMICO["example_parallel"]
resultado = ejecutar_consulta(query)

df = pd.DataFrame(resultado)
print("Volumen de negocio por país:")
print(df.to_string(index=False))

## 7. Exportación de Datos

Exportar resultados de análisis a JSON o CSV.

In [ ]:
# Exportar empresas de alto riesgo a CSV
query = """
CALL apoc.export.csv.query(
    'MATCH (e:Empresa)
     WHERE e.es_fantasma = true
     OPTIONAL MATCH (e)-[f:EMITE_FACTURA]->()
     WITH e, sum(f.monto_total) AS volumen
     RETURN e.nombre AS empresa, e.nif AS nif, e.capital_social AS capital,
            e.empleados AS empleados, volumen
     ORDER BY volumen DESC',
    'empresas_alto_riesgo.csv'
)
YIELD file, rows
RETURN file, rows
"""
# // null,  // null en lugar de nombre de archivo
# // {stream: true}

try:
    resultado = ejecutar_consulta(query)
    print(f"✓ Exportado: {resultado[0]['file']}")
    print(f"  Filas: {resultado[0]['rows']}")
    print(f"  Filas: {resultado}")
except Exception as e:
    print(f"Nota: {e}")
    print("(La exportación requiere permisos en neo4j.conf)")

Add en neo4j.conf:

- apoc.export.file.enabled=true
- apoc.import.file.enabled=true

- dbms.security.allow_csv_import_from_file_urls=true

## 8. Scoring de Fraude con APOC

Usar funciones de APOC para cálculos complejos.

In [ ]:
# Scoring usando apoc.map y apoc.coll
query = APOC_SCORING_FRAUDE["query"]
resultado = ejecutar_consulta(query)

df = pd.DataFrame(resultado)
print("\nTop 10 empresas por score de riesgo (calculado con APOC):")
print(df[['empresa', 'score_total', 'nivel_riesgo', 'num_facturas', 'volumen']].head(10).to_string(index=False))

## 9. Resumen y Ejercicios

### Lo que Aprendiste

✅ Metadatos y utilidades básicas de APOC  
✅ Expansión de caminos con `apoc.path`  
✅ Procesamiento en lotes con `apoc.periodic`  
✅ Consultas dinámicas con `apoc.cypher`  
✅ Exportación de datos  
✅ Scoring complejo con funciones APOC  

### Ejercicios Propuestos

1. **Path Expansion**: Encuentra todas las rutas > 3 saltos entre empresas fantasma
2. **Batch Update**: Marca como "revisado" todas las empresas con score > 50
3. **Export**: Exporta la red completa de empresas sospechosas a JSON
4. **Dynamic Query**: Crea una función que analice cualquier país pasado como parámetro
5. **Text Processing**: Usa `apoc.text.distance` para encontrar empresas con nombres similares

### Próximos Pasos

Continuar con **03_gds_plugins.ipynb** para aprender algoritmos avanzados de Graph Data Science.

### Recursos

- [Documentación APOC](https://neo4j.com/labs/apoc/)
- [APOC Cheat Sheet](https://neo4j.com/labs/apoc/4.4/overview/)
- Ver `consultas_apoc.py` para todos los ejemplos

In [ ]:
# Limpiar: Cerrar conexión
driver.close()
print("✓ Conexión cerrada")